In [3]:
!pip install TikTokApi
!playwright install
!playwright install-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.4/65.4 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 MB 24.0 MB/s eta 0:00:00
(node:2163) [DEP0169] DeprecationWarning: `url.parse()` behavior is not standardized and prone to errors that have security implications. Use the WHATWG URL API instead. CVEs are not issued for `url.parse()` vulnerabilities.
(Use `node --trace-deprecation ...` to show where the warning was created)
167.3 MiB [] 0% 0.0s167.3 MiB [] 0% 41.4s167.3 MiB [] 0% 17.5s167.3 MiB [] 0% 14.5s167.3 MiB [] 0% 11.7s167.3 MiB [] 1% 5.8s167.3 MiB [] 2% 4.6s167.3 MiB [] 2% 3.8s167.3 MiB [] 3% 3.5s167.3 MiB [] 3% 3.6s167.3 MiB [] 4% 3.5s167.3 MiB [] 5% 3.1s167.3 MiB [] 6% 2.8s167.3 MiB [] 7% 2.6s167.3 MiB [] 8% 2.5s167.3 MiB [] 9% 2.4s167.3 MiB [] 10% 2.3s167.3 MiB [] 11% 2.2s167.3 MiB [] 12% 2.1s167.3 MiB [] 13% 2.0s167.3 MiB [] 14% 2.0s167.3 MiB [] 15% 2.0s167.3 MiB [] 16% 2.0s167.3 MiB [] 17% 2.0s167.3 MiB [] 17% 1.9s167.3 MiB [] 18% 1.9s167.3 MiB

In [4]:
import asyncio
from TikTokApi import TikTokApi
import pandas as pd
from google.colab import files
import os

async def crawl_and_export_csv(keyword, video_count=10):
    hashtag_name = keyword.replace(" ", "").replace("#", "")
    print(f"Đang tìm kiếm với hashtag: #{hashtag_name}")

    api = TikTokApi()
    video_data_list = []

    try:
        # Khởi tạo session
        await api.create_sessions(
            num_sessions=1,
            sleep_after=10,
            browser='webkit',
            headless=True
        )

        tag = api.hashtag(name=hashtag_name)
        count = 0
        print("Đang lấy dữ liệu... (vui lòng chờ)")

        async for video in tag.videos(count=video_count):
            v_info = video.as_dict
            author = v_info.get("author", {}).get("uniqueId")
            v_id = v_info.get("id")
            stats = v_info.get("stats", {})

            # Thu thập các trường thông tin quan trọng
            record = {
                "Video ID": v_id,
                "URL": f"https://www.tiktok.com/@{author}/video/{v_id}",
                "Tác giả": author,
                "Mô tả": v_info.get("desc", ""),
                "Lượt tim": stats.get("diggCount", 0),
                "Lượt xem": stats.get("playCount", 0),
                "Lượt chia sẻ": stats.get("shareCount", 0),
                "Lượt bình luận": stats.get("commentCount", 0)
            }
            video_data_list.append(record)

            print(f"- Đã lấy video {count + 1}: {record['URL']}")

            count += 1
            if count >= video_count:
                break

        if video_data_list:
            # Tạo DataFrame và xuất CSV
            df = pd.DataFrame(video_data_list)
            csv_filename = f"tiktok_data_{hashtag_name}.csv"
            # utf-8-sig giúp Excel hiển thị đúng tiếng Việt
            df.to_csv(csv_filename, index=False, encoding='utf-8-sig')

            print(f"\n[XONG] Đã thu thập {len(video_data_list)} video.")
            print(f"Đang tự động tải file '{csv_filename}' xuống...")
            files.download(csv_filename)
        else:
            print("\n[THÔNG BÁO] Không có dữ liệu để xuất file (có thể do bị chặn IP).")

    except Exception as e:
        print(f"Lỗi: {e}")
    finally:
        await api.close_sessions()

# --- CẤU HÌNH CHẠY ---
keyword = "xe hơi" # @param {type:"string"}
video_limit = 20 # @param {type:"integer"}

await crawl_and_export_csv(keyword, video_count=video_limit)

Đang tìm kiếm với hashtag: #xehơi
Đang lấy dữ liệu... (vui lòng chờ)
- Đã lấy video 1: https://www.tiktok.com/@zoom6car/video/7613253519184842005
- Đã lấy video 2: https://www.tiktok.com/@anhchuxuongdeptrai/video/7612659344768896264
- Đã lấy video 3: https://www.tiktok.com/@vungocthanhh/video/7353136888313679124
- Đã lấy video 4: https://www.tiktok.com/@quenha.6/video/7608966011928317215
- Đã lấy video 5: https://www.tiktok.com/@cuongnuocmy/video/7427307108728687915
- Đã lấy video 6: https://www.tiktok.com/@cuongnuocmy/video/7377548592749317418
- Đã lấy video 7: https://www.tiktok.com/@autocarclub/video/7275932066582154503
- Đã lấy video 8: https://www.tiktok.com/@thaidacde1999/video/7544259066764479762
- Đã lấy video 9: https://www.tiktok.com/@sieuxe.360/video/7612687977428339975
- Đã lấy video 10: https://www.tiktok.com/@gatgu_channel/video/7456391235574517008
- Đã lấy video 11: https://www.tiktok.com/@chuyen_ve_oto/video/7605222950781898004
- Đã lấy video 12: https://www.tiktok.com/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [5]:
#Tách lấy cột URL
import pandas as pd
from google.colab import files
import io

# 1. Tải file CSV lên từ máy tính
print("Hãy chọn file CSV bạn vừa tải xuống:")
uploaded = files.upload()

# 2. Đọc file vừa tải lên
for filename in uploaded.keys():
    df_imported = pd.read_csv(io.BytesIO(uploaded[filename]))

    # 3. Kiểm tra và lấy cột URL
    if 'URL' in df_imported.columns:
        url_column = df_imported[['URL']]
        print(f"\n--- Đã trích xuất cột URL từ file: {filename} ---")
        display(url_column)

        # 4. Lưu thành file mới và tự động tải về
        output_filename = f"extracted_urls_{filename}"
        url_column.to_csv(output_filename, index=False, encoding='utf-8-sig')

        print(f"Đang tự động tải file kết quả '{output_filename}' xuống...")
        files.download(output_filename)
    else:
        print(f"\n[LỖI] Không tìm thấy cột 'URL' trong file {filename}.")
        print("Các cột hiện có:", df_imported.columns.tolist())

Hãy chọn file CSV bạn vừa tải xuống:


Saving tiktok_data_xehơi.csv to tiktok_data_xehơi (1).csv

--- Đã trích xuất cột URL từ file: tiktok_data_xehơi (1).csv ---


,URL
0,https://www.tiktok.com/@zoom6car/video/7613253...
1,https://www.tiktok.com/@anhchuxuongdeptrai/vid...
2,https://www.tiktok.com/@vungocthanhh/video/735...
3,https://www.tiktok.com/@quenha.6/video/7608966...
4,https://www.tiktok.com/@cuongnuocmy/video/7427...
5,https://www.tiktok.com/@cuongnuocmy/video/7377...
6,https://www.tiktok.com/@autocarclub/video/7275...
7,https://www.tiktok.com/@thaidacde1999/video/75...
8,https://www.tiktok.com/@sieuxe.360/video/76126...
9,https://www.tiktok.com/@gatgu_channel/video/74...


Đang tự động tải file kết quả 'extracted_urls_tiktok_data_xehơi (1).csv' xuống...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
#Tổng hợp công đoạn crawl
import pandas as pd
import requests
import re
import time
import random
from google.colab import files

def run_full_comment_crawl(target_df, comments_limit=30):
    all_comments_data = []
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        "Referer": "https://www.tiktok.com/"
    }

    # 1. Trích xuất URL
    if 'URL' not in target_df.columns:
        print("Lỗi: Không tìm thấy cột 'URL' trong dữ liệu.")
        return

    urls = target_df['URL'].tolist()
    print(f"Bắt đầu quét {len(urls)} video...")

    # 2. Duyệt qua từng URL và lấy Comment
    for i, url in enumerate(urls):
        try:
            match = re.search(r'video/(\d+)', url)
            if not match: continue
            video_id = match.group(1)

            print(f"[{i+1}/{len(urls)}] Đang lấy comment video ID: {video_id}...")
            api_url = f"https://www.tiktok.com/api/comment/list/?aid=1988&aweme_id={video_id}&count={comments_limit}&cursor=0"

            response = requests.get(api_url, headers=headers, timeout=15)

            if response.status_code == 200:
                data = response.json()
                comments = data.get('comments', [])

                for c in comments:
                    all_comments_data.append({
                        "Video ID": video_id,
                        "Comment Content": c.get("text", ""),
                        "User": c.get("user", {}).get("unique_id", ""),
                        "Like Count": c.get("digg_count", 0)
                    })
                print(f"  -> Thành công: Lấy được {len(comments)} bình luận.")
            else:
                print(f"  -> Thất bại: Mã lỗi {response.status_code}")

            # Nghỉ giữa các lần gọi để tránh bị chặn
            time.sleep(random.uniform(1.5, 3.0))

        except Exception as e:
            print(f"  -> Lỗi khi xử lý URL {url}: {e}")
            continue

    # 3. Xuất file và tải về
    if all_comments_data:
        df_result = pd.DataFrame(all_comments_data)
        output_name = "tiktok_all_comments.csv"
        df_result.to_csv(output_name, index=False, encoding='utf-8-sig')
        print(f"\n--- HOÀN THÀNH ---")
        print(f"Tổng số bình luận thu thập: {len(df_result)}")
        print(f"Đang tải file '{output_name}'...")
        files.download(output_name)
    else:
        print("\nKhông thu thập được bình luận nào.")

# Kiểm tra nếu biến df_imported (từ bước upload) tồn tại
if 'df_imported' in globals():
    run_full_comment_crawl(df_imported, comments_limit=30)
else:
    print("Vui lòng chạy ô 'Tải file CSV lên' trước khi thực hiện bước này.")

Bắt đầu quét 20 video...
[1/20] Đang lấy comment video ID: 7613253519184842005...
  -> Thành công: Lấy được 1 bình luận.
[2/20] Đang lấy comment video ID: 7612659344768896264...
  -> Thành công: Lấy được 3 bình luận.
[3/20] Đang lấy comment video ID: 7353136888313679124...
  -> Thành công: Lấy được 29 bình luận.
[4/20] Đang lấy comment video ID: 7608966011928317215...
  -> Thành công: Lấy được 28 bình luận.
[5/20] Đang lấy comment video ID: 7427307108728687915...
  -> Thành công: Lấy được 29 bình luận.
[6/20] Đang lấy comment video ID: 7377548592749317418...
  -> Thành công: Lấy được 29 bình luận.
[7/20] Đang lấy comment video ID: 7275932066582154503...
  -> Thành công: Lấy được 29 bình luận.
[8/20] Đang lấy comment video ID: 7544259066764479762...
  -> Thành công: Lấy được 28 bình luận.
[9/20] Đang lấy comment video ID: 7612687977428339975...
  -> Thành công: Lấy được 1 bình luận.
[10/20] Đang lấy comment video ID: 7456391235574517008...
  -> Thành công: Lấy được 30 bình luận.
[11/20]

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>